# Bi-Directional LSTM Poem Generator

This notebook is adapted from `Reema-Khaseeb/NLP-poem-generator` and keeps the same core model architecture:

- `Embedding`
- two stacked `Bidirectional(LSTM(..., return_sequences=True))` layers
- `BatchNormalization` + `Dropout`
- one final `LSTM(300)` layer
- dense regularized projection
- final softmax over vocabulary

Upload this notebook and `poem.txt` to Google Colab, then run the cells from top to bottom. The notebook expects `poem.txt` in the current Colab working directory (`/content/poem.txt`).

Safety note: this notebook does not run shell commands, download files, mount drives, read secrets, or load pickle files. It only reads `poem.txt` and writes local training artifacts.


In [ ]:
from pathlib import Path

DATA_PATH = Path('poem.txt')
if not DATA_PATH.exists():
    raise FileNotFoundError(
        'poem.txt was not found. Upload poem.txt to the Colab file browser, '
        'then rerun this cell.'
    )

print(f'Using data file: {DATA_PATH.resolve()}')


In [ ]:
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import tensorflow.keras.utils as ku
from sklearn.model_selection import train_test_split
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import BatchNormalization, Bidirectional, Dense, Dropout, Embedding, LSTM
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
# Read and lightly normalize the source poem corpus.
data = DATA_PATH.read_text(encoding='utf-8')
corpus = [line.strip().lower() for line in data.splitlines() if line.strip()]

print(f'Characters: {len(data):,}')
print(f'Non-empty lines: {len(corpus):,}')
print('Sample lines:')
for line in corpus[:5]:
    print('  ', line)


In [ ]:
# Tokenizer and n-gram sequence creation, matching the source notebook approach.
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

vocab_size = len(tokenizer.word_index) + 1
print('Vocabulary size:', vocab_size)

input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i + 1]
        input_sequences.append(n_gram_sequence)

if not input_sequences:
    raise ValueError('No training sequences were created. Check poem.txt contents.')

max_sequence_len = max(len(seq) for seq in input_sequences)
input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')
)

predictors = input_sequences[:, :-1]
labels = input_sequences[:, -1]
labels = ku.to_categorical(labels, num_classes=vocab_size)
sequence_len = predictors.shape[1]

print('Max sequence length:', max_sequence_len)
print('Predictors shape:', predictors.shape)
print('Labels shape:', labels.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    predictors,
    labels,
    test_size=0.25,
    random_state=SEED,
)

print('Train:', X_train.shape, y_train.shape)
print('Test: ', X_test.shape, y_test.shape)


In [ ]:
def create_bi_lstm_model(vocab_size: int, sequence_len: int) -> Sequential:
    """Create the Bi-Directional LSTM architecture from the source repo."""
    model = Sequential(name='reema_bi_lstm_poem_generator')
    model.add(
        Embedding(
            input_dim=vocab_size,
            output_dim=sequence_len,
            input_length=sequence_len,
        )
    )

    model.add(Bidirectional(LSTM(units=150, return_sequences=True)))

    model.add(Bidirectional(LSTM(units=150, return_sequences=True)))
    model.add(BatchNormalization())
    model.add(Dropout(0.25))

    model.add(LSTM(units=300))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))

    # The source notebook used `total_words + 1/2`, which becomes a float in Python.
    # This keeps the intended half-vocabulary projection while making the layer valid.
    model.add(
        Dense(
            max(64, vocab_size // 2),
            activation='relu',
            kernel_regularizer=regularizers.l2(0.01),
        )
    )
    model.add(BatchNormalization())
    model.add(Dropout(0.3))

    model.add(Dense(units=vocab_size, activation='softmax'))
    model.compile(
        loss='categorical_crossentropy',
        optimizer=Adam(learning_rate=0.001),
        metrics=['accuracy'],
    )
    return model


model = create_bi_lstm_model(vocab_size=vocab_size, sequence_len=sequence_len)
model.summary()


In [ ]:
EPOCHS = 150
BATCH_SIZE = 128
CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

callbacks = [
    ModelCheckpoint(
        filepath=str(CHECKPOINT_DIR / 'best_bi_lstm.weights.h5'),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_accuracy',
        patience=10,
        mode='max',
        factor=0.1,
        min_lr=1e-5,
        verbose=1,
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=25,
        verbose=1,
        mode='max',
        restore_best_weights=True,
    ),
]

history = model.fit(
    X_train,
    y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=2,
    validation_data=(X_test, y_test),
    callbacks=callbacks,
)


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history.history['loss'], 'r', label='train loss')
plt.plot(history.history['val_loss'], 'b', label='validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss')
plt.legend()
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(history.history['accuracy'], 'r', label='train accuracy')
plt.plot(history.history['val_accuracy'], 'b', label='validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy')
plt.legend()
plt.show()


In [ ]:
results = model.evaluate(X_test, y_test, verbose=1)
print(dict(zip(model.metrics_names, results)))

model.save('reema_bi_lstm_poem_generator.keras')
Path('tokenizer.json').write_text(tokenizer.to_json(), encoding='utf-8')
Path('training_config.json').write_text(
    json.dumps(
        {
            'vocab_size': vocab_size,
            'max_sequence_len': int(max_sequence_len),
            'sequence_len': int(sequence_len),
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'source_repo': 'https://github.com/Reema-Khaseeb/NLP-poem-generator/tree/main',
        },
        indent=2,
    ),
    encoding='utf-8',
)

print('Saved model: reema_bi_lstm_poem_generator.keras')
print('Saved tokenizer: tokenizer.json')
print('Saved config: training_config.json')


In [ ]:
def generate_text(model, tokenizer, sequence_len, seed_text, num_words):
    output_words = []
    input_text = seed_text

    for _ in range(num_words):
        encoded_text = tokenizer.texts_to_sequences([input_text])[0]
        padded_text = pad_sequences(
            [encoded_text],
            maxlen=sequence_len,
            truncating='pre',
            padding='pre',
        )
        predictions = model.predict(padded_text, verbose=0)[0]
        predicted_id = int(np.argmax(predictions))
        predicted_word = tokenizer.index_word.get(predicted_id, '')
        if not predicted_word:
            break

        input_text += ' ' + predicted_word
        output_words.append(predicted_word)

    return ' '.join(output_words)


seed_text = 'i looked behind'
print(seed_text, generate_text(model, tokenizer, sequence_len, seed_text, num_words=25))
